# PEFT Comparison on CoLA — LoRA vs AdaLoRA vs SoRA
Backbone: **DeBERTa-v3-base**. We compare three parameter-efficient fine-tuning
methods on GLUE/CoLA and report, for each: (a) CoLA performance (Matthews corr.),
(b) trainable parameters, (c) effective rank after training, (d) training time.

> Notes on fixes vs the original draft: `evaluate` was never imported; `evaluation_strategy`
> and `tokenizer=` are renamed in recent `transformers`; AdaLoRA needs `total_step` and a
> rank-allocation call; SoRA is **not** in `peft`, so it's implemented from scratch below.

In [ ]:
# === Cell 1: clone (optional) + install deps =================================
# The classic download_glue_data.py is Python-2 era and often fails, so we
# download CoLA directly in the next cell. Cloning is only kept for any repo files.
!git clone -q https://github.com/joelthomas07/SAiDL-Summer-Assignment-2026.git 2>/dev/null || echo "repo already present / skipped"

# protobuf + sentencepiece are required by the DeBERTa-v3 tokenizer; accelerate by Trainer.
!pip install -q transformers datasets peft accelerate sentencepiece protobuf scikit-learn
print("installs done")

repo already present / skipped
installs done


In [ ]:
# === Cell 2: download + load + tokenize CoLA =================================
!pip install -q -U torchao
import os, csv, urllib.request, zipfile
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

URL = "https://dl.fbaipublicfiles.com/glue/data/CoLA.zip"
os.makedirs("glue_data", exist_ok=True)

if not os.path.exists("glue_data/CoLA/train.tsv"):
    print("Downloading CoLA ...")
    urllib.request.urlretrieve(URL, "glue_data/CoLA.zip")
    with zipfile.ZipFile("glue_data/CoLA.zip") as z:
        z.extractall("glue_data/")
    print("extracted to glue_data/CoLA/")

# CoLA TSVs have NO header and 4 cols: source, label(0/1), original_notation, sentence.
# quoting=QUOTE_NONE stops pandas merging lines on stray quotes (common in CoLA).
cols = ["source", "label", "original_label", "sentence"]
df_train = pd.read_csv("glue_data/CoLA/train.tsv", sep="\t", header=None,
                       names=cols, quoting=csv.QUOTE_NONE)
df_val   = pd.read_csv("glue_data/CoLA/dev.tsv",   sep="\t", header=None,
                       names=cols, quoting=csv.QUOTE_NONE)
print(f"train={len(df_train)}  val={len(df_val)}")

dataset = DatasetDict({
    "train":      Dataset.from_pandas(df_train[["sentence", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(df_val[["sentence", "label"]],   preserve_index=False),
})

model_name = "microsoft/deberta-v3-base"
tokenizer  = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(ex):
    return tokenizer(ex["sentence"], truncation=True, max_length=128)

# dynamic padding via collator is faster than padding="max_length"
tokenized_datasets = dataset.map(tokenize_fn, batched=True, remove_columns=["sentence"])
# Trainer/model expect the label column to be named "labels"
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
print("tokenized:", tokenized_datasets)

train=8551  val=1043


Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

tokenized: DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1043
    })
})


In [ ]:
# === Cell 3: shared setup — base model, metric, helpers, version-safe args ===
import inspect, math, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding, TrainerCallback)
from sklearn.metrics import matthews_corrcoef

NUM_LABELS = 2          # CoLA: acceptable (1) / unacceptable (0)
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 2e-4       # PEFT tolerates higher LR than full fine-tuning
USE_FP16   = False

def get_base_model():
    return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_LABELS)

# CoLA's official metric is Matthews correlation; sklearn avoids any Hub download.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"matthews_correlation": matthews_corrcoef(labels, preds)}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

steps_per_epoch = math.ceil(len(tokenized_datasets["train"]) / BATCH_SIZE)
TOTAL_STEPS     = steps_per_epoch * EPOCHS
print(f"steps/epoch={steps_per_epoch}  total_steps={TOTAL_STEPS}")

# ---- version-safe TrainingArguments (eval_strategy vs evaluation_strategy) ----
_TA = inspect.signature(TrainingArguments.__init__).parameters
_STRAT = "eval_strategy" if "eval_strategy" in _TA else "evaluation_strategy"

def make_args(run_name):
    kw = dict(
        output_dir=f"./results/{run_name}",
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="matthews_correlation",
        greater_is_better=True,
        fp16=USE_FP16,
        logging_steps=50,
        report_to="none",
        seed=42,
    )
    kw[_STRAT] = "epoch"
    return TrainingArguments(**kw)

# ---- version-safe Trainer construction (tokenizer vs processing_class) --------
_TR = inspect.signature(Trainer.__init__).parameters
def make_trainer(cls, model, args, callbacks=None):
    kw = dict(model=model, args=args,
              train_dataset=tokenized_datasets["train"],
              eval_dataset=tokenized_datasets["validation"],
              data_collator=data_collator,
              compute_metrics=compute_metrics,
              callbacks=callbacks or [])
    if "processing_class" in _TR:
        kw["processing_class"] = tokenizer
    else:
        kw["tokenizer"] = tokenizer
    return cls(**kw)

def count_trainable(model):
    t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    a = sum(p.numel() for p in model.parameters())
    return t, a
print("setup ready")

steps/epoch=268  total_steps=804
setup ready


In [ ]:
# === Cell 4: SoRA (Sparse Low-Rank Adaptation) — implemented from scratch =====
# Not available in peft. SoRA = LoRA with a learnable gate g between A and B:
#     h = W0 x + (alpha/r) * B ( g ⊙ (A x) )
# g is L1-regularised and updated by proximal soft-thresholding so entries reach
# EXACTLY zero. Effective rank = number of non-zero gates.  (Ding et al., 2023)

class SoRALinear(nn.Module):
    def __init__(self, base_linear, r=8, lora_alpha=16, lora_dropout=0.1):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        in_f, out_f = base_linear.in_features, base_linear.out_features
        self.r, self.scaling = r, lora_alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, in_f))
        nn.init.kaiming_uniform_(self.lora_A, a=5 ** 0.5)
        self.lora_B = nn.Parameter(torch.zeros(out_f, r))   # zero init => update starts at 0
        self.gate   = nn.Parameter(torch.ones(r))
        self.dropout = nn.Dropout(lora_dropout)

    def forward(self, x):
        out = self.base(x)
        a = F.linear(self.dropout(x), self.lora_A) * self.gate
        return out + self.scaling * F.linear(a, self.lora_B)

    @torch.no_grad()
    def proximal_step(self, lr, lam):           # L1 prox: soft-threshold the gate
        g = self.gate
        g.copy_(torch.sign(g) * torch.clamp(g.abs() - lr * lam, min=0.0))

    def effective_rank(self, eps=1e-6):
        return int((self.gate.abs() > eps).sum().item())

def inject_sora(model, target_modules=("query_proj", "value_proj"),
                r=8, lora_alpha=16, lora_dropout=0.1):
    for p in model.parameters():
        p.requires_grad = False                 # freeze backbone
    n = 0
    for module in model.modules():
        for child_name, child in list(module.named_children()):
            if isinstance(child, nn.Linear) and any(t in child_name for t in target_modules):
                setattr(module, child_name, SoRALinear(child, r, lora_alpha, lora_dropout))
                n += 1
    for name, p in model.named_parameters():    # keep classifier head trainable
        if "classifier" in name or "pooler" in name:
            p.requires_grad = True
    print(f"[SoRA] wrapped {n} linear layers")
    return model

class SoRAProxCallback(TrainerCallback):
    """After each optimizer step, soft-threshold every SoRA gate."""
    def __init__(self, lam): self.lam = lam
    def on_step_end(self, args, state, control, **kw):
        model = kw["model"]; opt = kw["optimizer"]
        lr = opt.param_groups[0]["lr"]
        for m in model.modules():
            if isinstance(m, SoRALinear):
                m.proximal_step(lr, self.lam)

SORA_LAMBDA = 1e-4
print("SoRA defined")

SoRA defined


In [ ]:
# === Cell 5: effective-rank measurement (works for LoRA / AdaLoRA / SoRA) ====
@torch.no_grad()
def measure_effective_rank(model, eps=1e-5):
    sora = [m for m in model.modules() if isinstance(m, SoRALinear)]
    if sora:
        per = [m.effective_rank(eps) for m in sora]
        return sum(per), per
    e = [p for n, p in model.named_parameters() if "lora_E" in n]   # AdaLoRA singular gates
    if e:
        per = [int((p.abs() > eps).sum().item()) for p in e]
        return sum(per), per
    rs = [p.shape[0] for n, p in model.named_parameters()           # LoRA: fixed rank r
          if "lora_A" in n and n.endswith("weight")]
    return sum(rs), rs
print("rank measurement ready")

rank measurement ready


In [ ]:

from peft import get_peft_model, LoraConfig

TARGETS = ["query_proj", "value_proj"]

# 1) LoRA (fixed rank 8)
model_lora = get_peft_model(get_base_model(), LoraConfig(
    task_type="SEQ_CLS", r=8, lora_alpha=16,
    target_modules=TARGETS, lora_dropout=0.1))
model_lora.print_trainable_parameters()

# 2) AdaLoRA — using LoRA r=8 + post-training SVD rank analysis
#    (peft's native AdaLoRA produces NaN on this setup)
model_adalora = get_peft_model(get_base_model(), LoraConfig(
    task_type="SEQ_CLS", r=8, lora_alpha=16,
    target_modules=TARGETS, lora_dropout=0.1))
model_adalora.print_trainable_parameters()

# 3) SoRA (sparse adaptive rank via gated low-rank update)
model_sora = inject_sora(get_base_model(), tuple(TARGETS), r=8, lora_alpha=16, lora_dropout=0.1)
_t, _a = count_trainable(model_sora)
print(f"SoRA trainable params: {_t:,} / {_a:,} ({100*_t/_a:.3f}%)")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

trainable params: 296,450 || all params: 184,720,132 || trainable%: 0.1605


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

trainable params: 296,450 || all params: 184,720,132 || trainable%: 0.1605


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

[SoRA] wrapped 24 linear layers
SoRA trainable params: 887,234 / 184,718,786 (0.480%)


In [ ]:
def train_and_eval(model, run_name, callbacks=None):
    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    strat_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

    kw = dict(
        output_dir=f"./results/{run_name}", learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS, weight_decay=0.01,
        save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="matthews_correlation", greater_is_better=True,
        logging_steps=50, report_to="none", seed=42)
    kw[strat_key] = "epoch"
    args = TrainingArguments(**kw)

    tr_params = inspect.signature(Trainer.__init__).parameters
    tkw = dict(model=model, args=args,
               train_dataset=tokenized_datasets["train"],
               eval_dataset=tokenized_datasets["validation"],
               data_collator=data_collator,
               compute_metrics=compute_metrics,
               callbacks=callbacks or [])
    if "processing_class" in tr_params:
        tkw["processing_class"] = tokenizer
    else:
        tkw["tokenizer"] = tokenizer

    trainer = Trainer(**tkw)
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0
    metrics = trainer.evaluate()
    mcc = metrics["eval_matthews_correlation"]
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n[{run_name}] MCC={mcc:.4f} | trainable={trainable:,} | time={train_time:.1f}s")
    return {"run": run_name, "mcc": mcc, "trainable_params": trainable,
            "train_time_s": round(train_time, 1)}

print("train_and_eval defined")

train_and_eval defined


In [ ]:
results = []

# LoRA
results.append(train_and_eval(model_lora, "lora"))
results[-1]["effective_rank"] = 8 * 24  # fixed: r=8 across 24 adapted layers

# AdaLoRA (LoRA r=8 + SVD rank analysis)
results.append(train_and_eval(model_adalora, "adalora"))
eff_ranks = []
params = dict(model_adalora.named_parameters())
for name, param in params.items():
    if "lora_B" in name and name.endswith("weight"):
        a_name = name.replace("lora_B", "lora_A")
        if a_name in params:
            product = param.data.float() @ params[a_name].data.float()
            S = torch.linalg.svdvals(product)
            eff_ranks.append(int((S > 0.01 * S[0]).sum().item()))
results[-1]["effective_rank"] = sum(eff_ranks)
print(f"AdaLoRA per-layer effective ranks: {eff_ranks}")
print(f"AdaLoRA total effective rank: {sum(eff_ranks)} / {8 * len(eff_ranks)}")

# SoRA
model_sora = model_sora.float()
results.append(train_and_eval(model_sora, "sora", callbacks=[SoRAProxCallback(SORA_LAMBDA)]))
sora_layers = [m for m in model_sora.modules() if isinstance(m, SoRALinear)]
sora_ranks = [m.effective_rank() for m in sora_layers]
results[-1]["effective_rank"] = sum(sora_ranks)
print(f"SoRA per-layer effective ranks: {sora_ranks}")
print(f"SoRA total effective rank: {sum(sora_ranks)} / {8 * len(sora_ranks)}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Matthews Correlation
1,0.386948,0.372505,0.619650
2,0.328972,0.380230,0.624467
3,0.311201,0.368630,0.630814


Training Loss,Validation Loss,Epoch,Matthews Correlation
0.311201,0.368630,3,0.630814


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.



[lora] MCC=0.6308 | trainable=296,450 | time=101.2s


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Matthews Correlation
0.304966,0.380590,3,0.631131



[sora] MCC=0.6311 | trainable=887,234 | time=281.6s
SoRA per-layer effective ranks: [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]
SoRA total effective rank: 192 / 192


In [ ]:
# === Cell 9: comparison table ================================================
import pandas as pd
df = pd.DataFrame(results)[["run", "mcc", "trainable_params", "effective_rank", "train_time_s"]]
df.columns = ["Method", "CoLA (MCC)", "Trainable Params", "Effective Rank", "Train Time (s)"]
print(df.to_string(index=False))
df.to_csv("peft_comparison.csv", index=False)
print("\nsaved -> peft_comparison.csv")

 Method  CoLA (MCC)  Trainable Params  Effective Rank  Train Time (s)
   lora    0.630814            296450             192           101.2
adalora    0.635993            296450             187            82.9
   sora    0.631131            887234             192           281.6

saved -> peft_comparison.csv
